# Przygotowanie danych EGMS – rejon KWB Turów

Notatnik porządkuje dane przycięte w QGIS (folder `dane_EGMS/`), tak żeby miały taki sam format jak dane referencyjne z Bytomia i dały się użyć w notatnikach `EGMS_analysis_pipeline.ipynb` oraz `EGMS_buildings_analysis.ipynb`.

**Obszar:** Zittau (DE) – Hrádek nad Nisou (CZ) – Bogatynia (PL), otoczenie odkrywki Turów.

Poprawiane problemy:

1. Punkty z dwóch kafli EGMS są zapisane **obok siebie** (kolumny bez końcówki i z końcówką `_2`), a nie jeden pod drugim.
2. Szereg czasowy zawiera **akwizycje zimowe**, które w danych referencyjnych zostały usunięte (śnieg zaburza pomiar InSAR).
3. Budynki z trzech krajów są zapisane w **trzech blokach kolumn** (CZ, PL, DE), a budynki na styku warstw są **pocięte na kawałki**.

| Wejście | Wyjście |
|---|---|
| `dane_EGMS/U.csv` | `EGMS_Turow_U.csv` |
| `dane_EGMS/E1.csv` | `EGMS_Turow_E.csv` |
| `dane_EGMS/budynki/budynki.shp` | `budynki_Turow.gpkg` |
| `daty_bez_zimy.txt` | |

In [1]:
import pandas as pd
import geopandas as gpd

# 1. Punkty EGMS

Każdy wiersz ma wypełnioną tylko jedną połowę kolumn: punkty z kafla północnego (E46N31) są w kolumnach bez końcówki, a punkty z kafla południowego (E46N30) w kolumnach z końcówką `_2`. Rozdzielamy obie połowy i układamy je jedna pod drugą.

Z szeregu czasowego zostawiamy tylko daty z pliku `daty_bez_zimy.txt` – to te same 239 dat co w danych z Bytomia (bez okresu ok. 22 grudnia – 5 marca).

In [2]:
daty = open("daty_bez_zimy.txt").read().split()

print("Liczba dat:", len(daty))
print("Pierwsza data:", daty[0])
print("Ostatnia data:", daty[-1])

Liczba dat: 239
Pierwsza data: 20200308
Ostatnia data: 20241206


In [3]:
def uporzadkuj_egms(plik_wejsciowy, plik_wyjsciowy):
    df = pd.read_csv(plik_wejsciowy, low_memory=False)

    # Kolumny pierwszej połowy (bez końcówki "_2")
    kolumny = [col for col in df.columns if not col.endswith("_2")]

    # Druga połowa dostaje takie same nazwy kolumn jak pierwsza
    polowa_1 = df[kolumny]
    polowa_2 = df[[col + "_2" for col in kolumny]]
    polowa_2.columns = kolumny

    # Połowy jedna pod drugą, bez pustych wierszy
    df = pd.concat([polowa_1, polowa_2]).dropna(subset=["pid"])

    # Atrybuty punktu + daty bez zimy
    atrybuty = [col for col in kolumny if not col.isdigit()]
    df = df[atrybuty + daty]

    df.to_csv(plik_wyjsciowy, index=False)

    print(plik_wyjsciowy)
    print("  Liczba punktów PS:", len(df))
    print("  Liczba kolumn:", len(df.columns))
    print("  Braki danych:", df.isna().sum().sum())

In [4]:
uporzadkuj_egms("dane_EGMS/U.csv", "EGMS_Turow_U.csv")
uporzadkuj_egms("dane_EGMS/E1.csv", "EGMS_Turow_E.csv")

EGMS_Turow_U.csv
  Liczba punktów PS: 6268
  Liczba kolumn: 253
  Braki danych: 0


EGMS_Turow_E.csv
  Liczba punktów PS: 6268
  Liczba kolumn: 253
  Braki danych: 0


# 2. Budynki

Warstwa `budynki.shp` (EUBUCCO) powstała z trzech warstw krajowych, ale kolumny każdego kraju są w osobnym bloku po 29 kolumn:

- CZ – kolumny bez końcówki (`id`, `type`, `height`, ...),
- PL – kolumny z końcówką `_2`,
- DE – kolumny z końcówką `_3`.

Każdy rekord ma wypełniony tylko blok swojego kraju. Rozdzielamy bloki i układamy je jeden pod drugim.

In [5]:
budynki = gpd.read_file("dane_EGMS/budynki/budynki.shp")

print("Liczba rekordów:", len(budynki))
print("Liczba kolumn:", len(budynki.columns))
print("CRS:", budynki.crs)

Liczba rekordów: 39183
Liczba kolumn: 88
CRS: EPSG:4326


In [6]:
# Nazwy kolumn z pierwszego bloku (w kolejnych blokach nazwy są obcięte do 10 znaków)
kolumny = list(budynki.columns[:29])

bloki = []
for i in range(3):
    blok = budynki.iloc[:, i * 29:(i + 1) * 29].copy()
    blok.columns = kolumny
    blok["geometry"] = budynki.geometry
    bloki.append(blok.dropna(subset=["id"]))

budynki = gpd.GeoDataFrame(pd.concat(bloki), geometry="geometry", crs=budynki.crs)

print("Liczba rekordów:", len(budynki))
print("Liczba unikalnych budynków (id):", budynki["id"].nunique())

Liczba rekordów: 39210
Liczba unikalnych budynków (id): 39156


## Sklejenie pociętych budynków

Kawałki tego samego budynku mają to samo `id`, więc łączymy je z powrotem w jeden obrys (`dissolve`). Zostawiamy najważniejsze atrybuty i dodajemy kolumnę `kraj`.

Na granicach państw ok. 25 budynków występuje w danych dwóch krajów (ten sam budynek z dwóch źródeł). Przy ok. 39 tys. budynków nie ma to wpływu na wyniki, więc ich nie usuwamy.

In [7]:
budynki = budynki[["id", "region_id", "type", "subtype", "height", "floors", "geometry"]]
budynki = budynki.dissolve(by="id", as_index=False)

# Kraj z kodu regionu: CZ, PL lub DE
budynki["kraj"] = budynki["region_id"].str[:2]

# Ten sam układ współrzędnych co dane EGMS
budynki = budynki.to_crs("EPSG:3035")

print("Liczba budynków:", len(budynki))
print(budynki["kraj"].value_counts())

Liczba budynków: 39156
kraj
DE    23101
PL     8799
CZ     7256
Name: count, dtype: int64


In [8]:
budynki.to_file("budynki_Turow.gpkg", layer="budynki", driver="GPKG")

print("Zapisano: budynki_Turow.gpkg")

Zapisano: budynki_Turow.gpkg


# Użycie w notatnikach analizy

- `EGMS_analysis_pipeline.ipynb` – zamiast `EGMS_Bytom.csv` wczytać `EGMS_Turow_U.csv` (albo `EGMS_Turow_E.csv` dla składowej wschód–zachód). Kolumny mają już pełne nazwy, więc `rename` niczego nie zmienia.
- `EGMS_buildings_analysis.ipynb` – zamiast `budynki_polaczone_all.gpkg` wczytać `budynki_Turow.gpkg`.